# Paso 2 -- Entrenar un PPO por cada recompensa (A, B, C, D)

Mismos hiperparámetros de PPO (`simulador/config/instance.yaml` ->
`agente.hiperparametros`, ya validados: `VecNormalize`, `ent_coef=0.01`,
`learning_rate=3e-4`, `n_steps=512`), mismo `total_timesteps` (corto, para
iterar), misma `semilla_entrenamiento` -- lo ÚNICO que cambia entre las 4
corridas es la recompensa (`EntornoRecompensaIntercambiable`, `src/`). Cada
modelo se guarda en su propia carpeta (`output/modelos/{A,B,C,D}/`).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../../simulador").resolve()
POLITICA_BASE_DIR = (BASE_DIR / "../../politica_base").resolve()
MODELO_RL_DIR = (BASE_DIR / "..").resolve()
BB_DIR = (BASE_DIR / "../../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(POLITICA_BASE_DIR / "src"))
sys.path.insert(0, str(MODELO_RL_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entorno_recompensa_intercambiable import EntornoRecompensaIntercambiable

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

cfg_propio = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_recompensas = cfg_propio["recompensas"]
cfg_entren_propio = cfg_propio["entrenamiento"]
cfg_agente = cfg_sim["agente"]
cfg_hiper = cfg_agente["hiperparametros"]
gamma = cfg_agente["gamma"]
cfg_escalon = cfg_sim["escalones"][cfg_entren_propio["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60

OUT_MODELOS = BASE_DIR / "output" / "modelos"
OUT_MODELOS.mkdir(parents=True, exist_ok=True)

total_timesteps = cfg_entren_propio["total_timesteps"]
semilla_entrenamiento = cfg_entren_propio["semilla_entrenamiento"]
print(f"Hiperparametros PPO (compartidos, de simulador/config): {cfg_hiper}")
print(f"gamma={gamma}, total_timesteps={total_timesteps}, semilla={semilla_entrenamiento}")


def construir_entorno(tipo_recompensa, semilla_entrenamiento):
    return EntornoRecompensaIntercambiable(
        tipo_recompensa=tipo_recompensa,
        cfg_recompensa_tipo=cfg_recompensas[tipo_recompensa],
        gamma=gamma,
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        semilla_entrenamiento=semilla_entrenamiento,
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        # cfg_recompensa de env.py -- no se usa (el wrapper descarta y recalcula), pero el
        # constructor de SimuladorBarcosBergen lo exige -- se pasa el de produccion, inerte.
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


Hiperparametros PPO (compartidos, de simulador/config): {'usar_vecnormalize': True, 'ent_coef': 0.01, 'learning_rate': 0.0003, 'n_steps': 512}
gamma=0.99, total_timesteps=30000, semilla=123


## Entrenar las 4, una por una

In [2]:
modelos_entrenados = {}
resultados_entrenamiento = []

import gc
import json

n_steps_usado = cfg_hiper.get("n_steps", 2048)

for tipo in ["A", "B", "C", "D"]:
    out_dir_tipo = OUT_MODELOS / tipo
    out_dir_tipo.mkdir(parents=True, exist_ok=True)

    # Si ya esta entrenado (modelo_ppo.zip guardado), no se repite -- se lee el tiempo
    # medido de metadata_entrenamiento.json si ya existe (corridas mas viejas, antes de
    # que este archivo existiera, no lo tienen -- ahi queda None, no inventado).
    if (out_dir_tipo / "modelo_ppo.zip").exists():
        df_monitor = pd.read_csv(out_dir_tipo / "monitor.monitor.csv", skiprows=1)
        y = df_monitor["r"].values
        meta_path = out_dir_tipo / "metadata_entrenamiento.json"
        dt_previo = json.load(open(meta_path, encoding="utf-8"))["tiempo_entrenamiento_s"] if meta_path.exists() else None
        resultados_entrenamiento.append({
            "tipo": tipo, "episodios": len(y), "tiempo_s": dt_previo,
            "reward_primeros_20": y[:20].mean(), "reward_ultimos_20": y[-20:].mean(),
        })
        modelos_entrenados[tipo] = out_dir_tipo
        print(f"[{tipo}] ya entrenado (se reusa) -- {len(y)} episodios, tiempo_s={dt_previo}")
        continue

    t0 = time.time()
    venv = DummyVecEnv([lambda t=tipo: Monitor(
        construir_entorno(t, semilla_entrenamiento=cfg_entren_propio["semilla_entrenamiento"]),
        filename=str(out_dir_tipo / "monitor"),
    )])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True, gamma=gamma)

    model = PPO(
        "MlpPolicy", venv, gamma=gamma,
        ent_coef=cfg_hiper.get("ent_coef", 0.0), learning_rate=cfg_hiper.get("learning_rate", 3e-4),
        n_steps=n_steps_usado, seed=cfg_entren_propio["semilla_entrenamiento"], verbose=0,
    )
    model.learn(total_timesteps=cfg_entren_propio["total_timesteps"])
    dt = time.time() - t0

    model.save(str(out_dir_tipo / "modelo_ppo"))
    venv.save(str(out_dir_tipo / "vecnormalize.pkl"))
    modelos_entrenados[tipo] = out_dir_tipo

    df_monitor = pd.read_csv(out_dir_tipo / "monitor.monitor.csv", skiprows=1)
    y = df_monitor["r"].values
    primeros20, ultimos20 = y[:20].mean(), y[-20:].mean()
    resultados_entrenamiento.append({
        "tipo": tipo, "episodios": len(y), "tiempo_s": dt,
        "reward_primeros_20": primeros20, "reward_ultimos_20": ultimos20,
    })
    print(f"[{tipo}] {dt:.0f}s, {len(y)} episodios -- primeros 20 ep: {primeros20:.2f}, ultimos 20 ep: {ultimos20:.2f}")

    # Metadata fija de esta corrida (tiempo, actualizaciones de PPO, pasos/seg) -- en su
    # propio archivo, no solo en el print ni en el CSV agregado de mas abajo.
    metadata_tipo = {
        "tipo_recompensa": tipo,
        "total_timesteps": cfg_entren_propio["total_timesteps"],
        "n_steps": n_steps_usado,
        "actualizaciones_ppo": cfg_entren_propio["total_timesteps"] // n_steps_usado,
        "tiempo_entrenamiento_s": dt,
        "tiempo_entrenamiento_min": dt / 60,
        "episodios": len(y),
        "pasos_por_segundo": cfg_entren_propio["total_timesteps"] / dt,
        "semilla_entrenamiento": cfg_entren_propio["semilla_entrenamiento"],
    }
    with open(out_dir_tipo / "metadata_entrenamiento.json", "w", encoding="utf-8") as f:
        json.dump(metadata_tipo, f, indent=2, ensure_ascii=False)

    # Liberar el modelo/entorno de esta iteracion antes de la siguiente -- sin esto, en una
    # corrida anterior el entrenamiento de C se volvio mucho mas lento que A/B en la misma
    # sesion de kernel (aislado en un proceso nuevo, C entrena a velocidad normal -- ver
    # README seccion 3), consistente con presion de memoria acumulada entre iteraciones.
    del model, venv
    gc.collect()

tabla_entrenamiento = pd.DataFrame(resultados_entrenamiento)
tabla_entrenamiento.to_csv(BASE_DIR / "output" / "comparacion" / "resumen_entrenamiento.csv", index=False)
display(tabla_entrenamiento.round(2))


[A] ya entrenado (se reusa) -- 335 episodios, tiempo_s=153.0087924003601
[B] ya entrenado (se reusa) -- 335 episodios, tiempo_s=126.72823667526245


[C] 187s, 335 episodios -- primeros 20 ep: -24866.20, ultimos 20 ep: -11072.93


[D] 227s, 335 episodios -- primeros 20 ep: -2803.34, ultimos 20 ep: -2108.63


,tipo,episodios,tiempo_s,reward_primeros_20,reward_ultimos_20
0,A,335,153.01,-9496.44,-8647.37
1,B,335,126.73,-24871.44,-12412.91
2,C,335,186.69,-24866.20,-11072.93
3,D,335,227.26,-2803.34,-2108.63


**Nota de lectura:** las columnas `reward_primeros_20`/`reward_ultimos_20`
son crudas, en la escala propia de CADA recompensa (A, con magnitudes
mucho más grandes que B/C/D por diseño -- ver Paso 1) -- sirven para
confirmar que cada entrenamiento efectivamente MEJORÓ con la práctica
(el número sube, sea cual sea su escala), NO para comparar A contra B/C/D
entre sí. Esa comparación (la que importa) es el Paso 3, sobre métricas
reales, no sobre estos rewards.

## Curvas de entrenamiento, las 4 juntas (cada una normalizada a su propio rango para poder verlas en el mismo gráfico)

In [3]:
import plotly.graph_objects as go

fig = go.Figure()
colores = {"A": "#2980b9", "B": "#27ae60", "C": "#8e44ad", "D": "#e67e22"}
for tipo in ["A", "B", "C", "D"]:
    df_monitor = pd.read_csv(OUT_MODELOS / tipo / "monitor.monitor.csv", skiprows=1)
    y = df_monitor["r"].values
    # Normalizado 0-1 dentro de su propio rango -- SOLO para poder comparar la FORMA
    # de la curva (¿converge?, ¿mejora rapido?) en el mismo grafico, no para comparar
    # valores absolutos entre tipos (eso nunca se hace -- ver nota de arriba).
    y_norm = (y - y.min()) / (y.max() - y.min() + 1e-9)
    media_movil = pd.Series(y_norm).rolling(20, min_periods=1).mean()
    fig.add_trace(go.Scatter(x=list(range(len(y))), y=media_movil, mode="lines",
                              line=dict(width=2.5, color=colores[tipo]), name=f"{tipo} (media movil 20 ep)"))
fig.update_layout(
    title="Curvas de entrenamiento, las 4 recompensas (reward normalizado 0-1 dentro de su propio rango)",
    xaxis_title="Episodio", yaxis_title="Reward normalizado (0=peor episodio propio, 1=mejor)",
    template="plotly_white",
)
fig.write_html(str(BASE_DIR / "output" / "comparacion" / "curvas_entrenamiento.html"))
fig.show()
